# 02 — Baselines and candidate selection

This notebook reviews models trained by `src/train.py`. Selection uses validation only: minimum RMSE for GPA, maximum enroll F1 for course outcome, and maximum macro-F1 for learning pace. The frozen test split is not used here.

In [ ]:
from pathlib import Path
import sys

COLAB_ROOT = Path('/content/SRG-Tracker')
PROJECT_ROOT = COLAB_ROOT if (COLAB_ROOT / 'src').is_dir() else (Path.cwd() if (Path.cwd() / 'src').is_dir() else Path.cwd().parent)
SRC_DIR = PROJECT_ROOT / 'src'
METRICS_DIR = PROJECT_ROOT / 'reports' / 'metrics'
MODELS_DIR = PROJECT_ROOT / 'models'
if not SRC_DIR.is_dir():
    raise FileNotFoundError(f'Cannot locate src directory from {Path.cwd()}')
sys.path.insert(0, str(PROJECT_ROOT)) if str(PROJECT_ROOT) not in sys.path else None

try:
    import json
    import pandas as pd
except ImportError as error:
    raise ImportError('Install project dependencies with: pip install -r requirements.txt') from error

To reproduce model training from the project root, run `python -m src.train`. It trains V1 reference, Dummy, linear, HistGradientBoosting, XGBoost, CatBoost, MLP-Adam, and raw-sequence GRU candidates.

In [ ]:
comparison_path = METRICS_DIR / 'validation_model_comparison.csv'
selection_path = MODELS_DIR / 'selection.json'
for required_path in (comparison_path, selection_path):
    if not required_path.exists():
        raise FileNotFoundError(f'Missing {required_path}. Run: python -m src.train')
comparison = pd.read_csv(comparison_path)
selection = json.loads(selection_path.read_text(encoding='utf-8'))
selection

In [ ]:
primary_metrics = {'gpa': 'rmse', 'outcome': 'f1_enroll', 'pace': 'macro_f1'}
rankings = {}
for task, metric in primary_metrics.items():
    path = METRICS_DIR / f'{task}_model_ranking.csv'
    if not path.exists():
        raise FileNotFoundError(f'Missing ranking table: {path}')
    columns = ['rank', 'model', 'representation', metric, 'model_size_bytes', 'train_seconds', 'median_inference_ms']
    rankings[task] = pd.read_csv(path)[columns]
rankings['gpa']

In [ ]:
rankings['outcome']

In [ ]:
rankings['pace']